# 025 — Switch Characterization Campaign (P6.8/P6.9)

Oscilloscope campaign on the CGC switch units (swB first, then swA) per
`raw/plans/2026-07-05-p6-cgc-explorer.md` section D. Runs on the P6.8
lab classes (`PSU`/`SW`/`SWHR` on `CGCDevice`) and the campaign tooling
in `devices.cgc.campaign`.

## SAFETY — non-negotiable (CGC email, notebook 023)

- Per-switch-channel dissipation **< 100 W** (pair <= 200 W).
- At 350 V: output current **<= 300 mA**.
- **Recipe order:** ramp voltage at **1 kHz** up to the target (<= 350 V)
  FIRST, **then** ramp frequency at full voltage.
- The `PSUWatchdog` polls PSU readback between steps: breach -> 10 %
  step-down, second consecutive breach -> outputs disabled. It is a SOFT
  watchdog — it does not replace watching the current readout.
- Broken sensors: **swA sensor 2, swB sensor 0** (skipped in housekeeping
  and plots).

## Wiring (notebook 023, authoritative)

| Switch | Outputs | PSU | Load |
|---|---|---|---|
| swB (SW)   | 0/1 | psu2 | Quadrupole 1 |
| swB (SW)   | 2/3 | psu1 | Ion Funnels |
| swA (SWHR) | 0/1 | psu3 | Quadrupole 2 |
| swA (SWHR) | 2/3 | psu4 | Quadrupoles 3/4 |

## SIM mode

`SIM = True` (default) builds every device with `test_mode=True`: no DLL,
no COM ports, all helpers run against the stateful sims — the P6.8
dry-run. Set `SIM = False` only at the bench, with Explorer closed
(one COM port, one owner) and the printed checklist from
`raw/guides/switch-operation-guide.md` on the table.

In [1]:
import sys
import os
import time
import logging
from datetime import datetime
from pathlib import Path

# --- The one switch that matters ---------------------------------------
SIM = True   # True: test_mode devices, zero hardware. False: THE BENCH.

# Add src to path
sys.path.append(os.path.join(os.getcwd(), '..', '..', 'src'))

from devices.cgc import PSU, SW, SWHR
from devices.cgc.campaign import (
    PSUWatchdog, WatchdogBreach, ramp_voltage_at_1khz, ramp_frequency,
    I_LIMIT_MA, P_LIMIT_W, RAMP_FREQ_KHZ, V_MAX,
)

# Canonical COM ports (lab_services/lab_config.toml [com_ports]); the
# fallback matches the table confirmed by the user 2026-07-03.
COM = {"PSU1": 15, "PSU2": 16, "PSU3": 17, "PSU4": 18, "swA": 9, "swB": 10}
try:
    import tomllib
    _cfg = Path(os.getcwd()) / ".." / ".." / ".." / "lab_services" / "lab_config.toml"
    with open(_cfg, "rb") as f:
        _ports = tomllib.load(f)["com_ports"]
    COM.update({k: _ports[k] for k in COM if k in _ports})
    print(f"COM ports from lab_config.toml: {COM}")
except Exception as e:
    print(f"lab_config.toml not readable ({e}) - using fallback table {COM}")

print(f"SIM = {SIM}   limits: {I_LIMIT_MA:.0f} mA / {P_LIMIT_W:.0f} W, "
      f"ramp at {RAMP_FREQ_KHZ:.0f} kHz, ceiling {V_MAX:.0f} V")

COM ports from lab_config.toml: {'PSU1': 15, 'PSU2': 16, 'PSU3': 17, 'PSU4': 18, 'swA': 9, 'swB': 10}
SIM = True   limits: 300 mA / 100 W, ramp at 1 kHz, ceiling 350 V


In [2]:
# Session logger: one file per campaign day + console.
logs_dir = Path(os.getcwd()) / ".." / "logs"
logs_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
logger = logging.getLogger(f"campaign_{stamp}")
logger.setLevel(logging.DEBUG)
if not logger.handlers:
    fh = logging.FileHandler(logs_dir / f"025_switch_campaign_{stamp}.log")
    fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)-7s %(message)s"))
    fh.setLevel(logging.DEBUG)
    logger.addHandler(fh)
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(logging.Formatter("%(levelname)-7s %(message)s"))
    console_handler.setLevel(logging.INFO)
    logger.addHandler(console_handler)
logger.info(f"campaign session start, SIM={SIM}")

INFO    campaign session start, SIM=True


## Devices

All six instruments; broken sensors skipped per unit.

In [3]:
psu1 = PSU("psu1", com=COM["PSU1"], port=0, logger=logger, test_mode=SIM)
psu2 = PSU("psu2", com=COM["PSU2"], port=1, logger=logger, test_mode=SIM)
psu3 = PSU("psu3", com=COM["PSU3"], port=2, logger=logger, test_mode=SIM)
psu4 = PSU("psu4", com=COM["PSU4"], port=3, logger=logger, test_mode=SIM)
swA = SWHR("swA", com=COM["swA"], stream=0, logger=logger, test_mode=SIM,
           skip_sensors=(2,))   # sensor 2 broken
swB = SW("swB", com=COM["swB"], port=0, logger=logger, test_mode=SIM,
         skip_sensors=(0,))     # sensor 0 broken

PSUS = [psu1, psu2, psu3, psu4]
ALL = [("psu1", psu1), ("psu2", psu2), ("psu3", psu3), ("psu4", psu4),
       ("swA", swA), ("swB", swB)]

# Feeding map (wiring table above): switch unit -> its two PSUs.
FEEDS = {"swB": (psu2, psu1), "swA": (psu3, psu4)}

INFO    [SIM] psu1    COM15  initialized (TEST MODE (simulated))


INFO    [SIM] psu2    COM16  initialized (TEST MODE (simulated))


INFO    [SIM] psu3    COM17  initialized (TEST MODE (simulated))


INFO    [SIM] psu4    COM18  initialized (TEST MODE (simulated))


INFO    [SIM] swA     COM9   initialized (TEST MODE (simulated))


INFO    [SIM] swB     COM10  initialized (TEST MODE (simulated))


In [4]:
# Connect everything + read-back check (temps prove the wire).
for name, dev in ALL:
    ok = dev.connect()
    print(f"{name:5s} connect: {ok}")
for name, dev in ALL:
    with dev.thread_lock:
        status, t0, t1, t2 = dev.get_sensor_data()
    print(f"{name:5s} temps: {t0:.1f} / {t1:.1f} / {t2:.1f} degC (status {status})")

INFO    [SIM] psu1    COM15  connected (simulated, no hardware)


INFO    [SIM] psu2    COM16  connected (simulated, no hardware)


INFO    [SIM] psu3    COM17  connected (simulated, no hardware)


INFO    [SIM] psu4    COM18  connected (simulated, no hardware)


INFO    [SIM] swA     COM9   connected (simulated, no hardware)


INFO    [SIM] swB     COM10  connected (simulated, no hardware)


psu1  connect: True
psu2  connect: True
psu3  connect: True
psu4  connect: True
swA   connect: True
swB   connect: True
psu1  temps: 29.8 / 33.5 / 33.2 degC (status 0)
psu2  temps: 33.3 / 30.0 / 33.8 degC (status 0)
psu3  temps: 27.0 / 34.7 / 29.4 degC (status 0)
psu4  temps: 26.5 / 32.0 / 27.4 degC (status 0)
swA   temps: 33.6 / 39.0 / 29.1 degC (status 0)
swB   temps: 28.2 / 25.5 / 26.5 degC (status 0)


## Watchdog

`dog.check()` polls every PSU output once; the ramp helpers call it after
every step. Leave a manual `dog.check()` in scope while probing so you can
poll between scope shots.

In [5]:
dog = PSUWatchdog(PSUS)
breaches = dog.check()
print("watchdog:", "ALL CLEAR" if not breaches else breaches)

watchdog: ALL CLEAR


## Live monitor (Bokeh, opens in Chrome)

Temps left, per-PSU V/I right, 300 mA safety line — the notebook-023
monitor trimmed onto the curated lab-layer reads (so it animates in SIM
too). Set `RUN_LIVE_MONITOR = True` and run the cell; stop it by
interrupting the kernel or leaving it False for headless dry-runs.

In [6]:
RUN_LIVE_MONITOR = False   # True at the bench (and for a sim look)
BOKEH_PORT = 5008          # 023 used 5007

if RUN_LIVE_MONITOR:
    import threading
    import webbrowser
    from datetime import timedelta
    from bokeh.plotting import figure
    from bokeh.layouts import column, row
    from bokeh.models import ColumnDataSource, DatetimeTickFormatter, Range1d, Span
    from bokeh.palettes import Category10
    from bokeh.server.server import Server
    from tornado.ioloop import IOLoop

    PLOT_SENSORS = {"psu1": (0, 1, 2), "psu2": (0, 1, 2), "psu3": (0, 1, 2),
                    "psu4": (0, 1, 2), "swA": (0, 1), "swB": (1, 2)}
    WINDOW_MIN, MAX_POINTS = 5, 7200
    colors = Category10[3]

    def make_document(doc):
        temp_sources = {n: ColumnDataSource(
            data={"time": [], **{f"t{s}": [] for s in PLOT_SENSORS[n]}})
            for n, _ in ALL}
        psu_sources = {n: ColumnDataSource(
            data={"time": [], "v0": [], "v1": [], "i0": [], "i1": []})
            for n, d in ALL if isinstance(d, PSU)}
        now = datetime.now()
        x_range = Range1d(start=now - timedelta(minutes=WINDOW_MIN), end=now)
        fmt = DatetimeTickFormatter(seconds="%H:%M:%S", minsec="%H:%M:%S",
                                    minutes="%H:%M:%S", hourmin="%H:%M:%S",
                                    hours="%H:%M:%S")
        temp_figs, psu_figs = [], []
        for n, _ in ALL:
            p = figure(title=f"Temp: {n}", x_axis_type="datetime", height=160,
                       width=850, x_range=x_range)
            for s in PLOT_SENSORS[n]:
                p.line("time", f"t{s}", source=temp_sources[n], line_width=2,
                       color=colors[s], legend_label=f"Sensor {s}")
            p.xaxis.formatter = fmt
            p.legend.location = "top_left"
            p.legend.click_policy = "hide"
            temp_figs.append(p)
        for n in psu_sources:
            pi = figure(title=f"{n}: V (solid axis left) / I", height=200,
                        width=850, x_axis_type="datetime", x_range=x_range)
            pi.line("time", "i0", source=psu_sources[n], line_width=2,
                    color=colors[0], legend_label="I ch0 [mA]")
            pi.line("time", "i1", source=psu_sources[n], line_width=2,
                    color=colors[1], legend_label="I ch1 [mA]")
            pi.add_layout(Span(location=I_LIMIT_MA, dimension="width",
                               line_color="red", line_dash="dashed"))
            pi.xaxis.formatter = fmt
            pi.legend.location = "top_left"
            pi.legend.click_policy = "hide"
            psu_figs.append(pi)

        def update():
            now = datetime.now()
            for n, dev in ALL:
                try:
                    with dev.thread_lock:
                        status, t0, t1, t2 = dev.get_sensor_data()
                    if status != dev.NO_ERR:
                        continue
                    temps = (t0, t1, t2)
                    temp_sources[n].stream(
                        {"time": [now],
                         **{f"t{s}": [temps[s]] for s in PLOT_SENSORS[n]}},
                        rollover=MAX_POINTS)
                except Exception:
                    pass
            for n, dev in [(n, d) for n, d in ALL if isinstance(d, PSU)]:
                try:
                    with dev.thread_lock:
                        s0, v0, i0, _ = dev.get_psu_data(dev.PSU_POS)
                        s1, v1, i1, _ = dev.get_psu_data(dev.PSU_NEG)
                    psu_sources[n].stream(
                        {"time": [now], "v0": [v0], "v1": [v1],
                         "i0": [i0 * 1000.0], "i1": [i1 * 1000.0]},
                        rollover=MAX_POINTS)
                except Exception:
                    pass
            x_range.start = now - timedelta(minutes=WINDOW_MIN)
            x_range.end = now

        doc.add_periodic_callback(update, 1000)
        doc.add_root(row(column(*temp_figs), column(*psu_figs)))
        doc.title = "025 switch campaign"

    def run_server():
        server = Server({"/": make_document}, port=BOKEH_PORT,
                        io_loop=IOLoop())
        server.start()
        server.io_loop.start()

    threading.Thread(target=run_server, daemon=True).start()
    time.sleep(1.5)
    webbrowser.open(f"http://localhost:{BOKEH_PORT}/")
    print(f"live monitor on http://localhost:{BOKEH_PORT}/")
else:
    print("live monitor disabled (RUN_LIVE_MONITOR = False)")

live monitor disabled (RUN_LIVE_MONITOR = False)


# swB session (first — its SwitchSym RF NVM ladder exists)

swB NVM: 1=Standby, 40=1 kHz ... 90=1 MHz, 101-106=1.2-5 MHz, 109 burst.
Timing formulas to verify on the scope:
`oscillator_period_ns = (Period_reg + 2) * 10`, `f = 100e6 / (Period_reg + 2)`.

Order per measurement point: switch config first, THEN PSU voltage
(ramped at 1 kHz), then frequency up. PSU current limits from 023:
300 mA (Q1 / psu2), 150 mA (funnels / psu1).

In [7]:
# Bring-up for the swB block: current limits + device enables.
psu2.set_psu0_output_current(300); psu2.set_psu1_output_current(300)  # Q1
psu1.set_psu0_output_current(150); psu1.set_psu1_output_current(150)  # IF
for p in (psu1, psu2):
    p.set_device_enable(True)
    p.set_psu_enable(True, True)
swB.set_device_enable(True)
dog.reset()
print("swB block armed:", dog.check() == [])

INFO    [SIM] psu2    COM16  setting PSU0 output current to 300.000 mA


INFO    [SIM] psu2    COM16  setting PSU1 output current to 300.000 mA


INFO    [SIM] psu1    COM15  setting PSU0 output current to 150.000 mA


INFO    [SIM] psu1    COM15  setting PSU1 output current to 150.000 mA


INFO    [SIM] psu1    COM15  setting device enable to True


INFO    [SIM] psu1    COM15  setting PSU enable to psu0=True, psu1=True


INFO    [SIM] psu2    COM16  setting device enable to True


INFO    [SIM] psu2    COM16  setting PSU enable to psu0=True, psu1=True


INFO    [SIM] swB     COM10  setting device enable to True


swB block armed: True


## M1 — frequency ladder vs `(reg+2)*10 ns`

Per rung: set frequency, note the period register, measure the true
period on the scope, fill the table in the guide. In SIM the loop just
proves the register arithmetic.

In [8]:
FREQ_LADDER_KHZ = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]
for f_khz in FREQ_LADDER_KHZ:
    swB.set_frequency_khz(f_khz)
    status, reg = swB.get_oscillator_period()
    f_calc = swB.CLOCK / (reg + swB.OSC_OFFSET)
    print(f"{f_khz:7.0f} kHz -> reg {reg:9d} -> calc {f_calc/1e3:9.3f} kHz "
          f"| period {(reg + 2) * 10:.0f} ns")
    if not SIM:
        input(f"  scope: measure period at {f_khz} kHz, note it, ENTER for next rung")
    dog.check()
swB.set_frequency_khz(RAMP_FREQ_KHZ)  # park at the safe ramp frequency

INFO    [SIM] swB     COM10  setting oscillator period to 99998 (~1000.0 Hz)


INFO    [SIM] swB     COM10  setting oscillator period to 49998 (~2000.0 Hz)


INFO    [SIM] swB     COM10  setting oscillator period to 19998 (~5000.0 Hz)


INFO    [SIM] swB     COM10  setting oscillator period to 9998 (~10000.0 Hz)


INFO    [SIM] swB     COM10  setting oscillator period to 4998 (~20000.0 Hz)


...

INFO    [SIM] swB     COM10  setting oscillator period to 198 (~500000.0 Hz)


INFO    [SIM] swB     COM10  setting oscillator period to 98 (~1000000.0 Hz)


INFO    [SIM] swB     COM10  setting oscillator period to 99998 (~1000.0 Hz)


      1 kHz -> reg     99998 -> calc     1.000 kHz | period 1000000 ns
      2 kHz -> reg     49998 -> calc     2.000 kHz | period 500000 ns
      5 kHz -> reg     19998 -> calc     5.000 kHz | period 200000 ns
     10 kHz -> reg      9998 -> calc    10.000 kHz | period 100000 ns
     20 kHz -> reg      4998 -> calc    20.000 kHz | period 50000 ns
     50 kHz -> reg      1998 -> calc    50.000 kHz | period 20000 ns
    100 kHz -> reg       998 -> calc   100.000 kHz | period 10000 ns
    200 kHz -> reg       498 -> calc   200.000 kHz | period 5000 ns
    500 kHz -> reg       198 -> calc   500.000 kHz | period 2000 ns
   1000 kHz -> reg        98 -> calc  1000.000 kHz | period 1000 ns


0

## M2 — duty cycle / pulse width + dead time

Width register: `Width_reg = duty * (Period_reg + 2) - 2`; delay minimum
= register 1 = 40 ns. Scope: verify width and the dead time between
complementary outputs at each duty point.

In [9]:
DUTY_POINTS = [0.2, 0.35, 0.5, 0.65, 0.8]
swB.set_frequency_khz(10)  # fixed 10 kHz for the duty sweep
for pulser in range(2):    # pulser 0 + 1 = the complementary pair on out 0/1
    swB.set_delay_minimum(pulser)
for duty in DUTY_POINTS:
    for pulser in range(2):
        swB.set_duty_cycle(pulser, duty)
    status, width = swB.get_pulser_width(0)
    print(f"duty {duty:4.2f} -> width register {width}")
    if not SIM:
        input("  scope: width + dead time, ENTER for next point")
    dog.check()
swB.set_frequency_khz(RAMP_FREQ_KHZ)

INFO    [SIM] swB     COM10  setting oscillator period to 9998 (~10000.0 Hz)


INFO    [SIM] swB     COM10  setting pulser 0 delay to 1


INFO    [SIM] swB     COM10  setting pulser 1 delay to 1


INFO    [SIM] swB     COM10  setting pulser 0 width to 1998


INFO    [SIM] swB     COM10  setting pulser 1 width to 1998


...

INFO    [SIM] swB     COM10  setting pulser 0 width to 7998


INFO    [SIM] swB     COM10  setting pulser 1 width to 7998


INFO    [SIM] swB     COM10  setting oscillator period to 99998 (~1000.0 Hz)


duty 0.20 -> width register 1998
duty 0.35 -> width register 3498
duty 0.50 -> width register 4998
duty 0.65 -> width register 6498
duty 0.80 -> width register 7998


0

## M3 — rise/fall vs amplitude (recipe ramp!)

Voltage ladder via `ramp_voltage_at_1khz` (watchdog after every step).
Scope: rise/fall time at each amplitude point on the loaded output.

In [10]:
V_POINTS = [50.0, 100.0, 150.0, 200.0, 250.0, 300.0, 350.0]
DWELL_S = 0.05 if SIM else 2.0
v_reached = 0.0
try:
    for v in V_POINTS:
        v_reached = ramp_voltage_at_1khz(
            swB, psu2, psu2.PSU_POS, v, step_v=10.0, dwell_s=DWELL_S,
            watchdog=dog)
        print(f"amplitude point {v_reached:.0f} V reached")
        if not SIM:
            input("  scope: rise/fall at this amplitude, ENTER for next")
except WatchdogBreach as e:
    print(f"RAMP HALTED: {e}")
print(f"final setpoint: {v_reached:.0f} V")

INFO    [SIM] swB     COM10  setting oscillator period to 99998 (~1000.0 Hz)


INFO    [SIM] psu2    COM16  setting PSU0 output voltage to 10.000 V


INFO    [SIM] psu2    COM16  setting PSU0 output voltage to 20.000 V


INFO    [SIM] psu2    COM16  setting PSU0 output voltage to 30.000 V


INFO    [SIM] psu2    COM16  setting PSU0 output voltage to 40.000 V


...

INFO    [SIM] psu2    COM16  setting PSU0 output voltage to 330.000 V


INFO    [SIM] psu2    COM16  setting PSU0 output voltage to 340.000 V


INFO    [SIM] psu2    COM16  setting PSU0 output voltage to 350.000 V


amplitude point 300 V reached


amplitude point 350 V reached
final setpoint: 350 V


## M4 — channel skew (2 probes)

Both probes on the complementary pair; measure edge-to-edge skew at 1 kHz
and at 100 kHz. Record in the guide table; no code beyond configuration.

In [11]:
swB.set_frequency_khz(RAMP_FREQ_KHZ)
print("skew point 1: 1 kHz (probes on out0/out1)")
if not SIM:
    input("  scope: skew at 1 kHz, ENTER to continue")
swB.set_frequency_khz(100)
print("skew point 2: 100 kHz")
if not SIM:
    input("  scope: skew at 100 kHz, ENTER to continue")
swB.set_frequency_khz(RAMP_FREQ_KHZ)
dog.check()

INFO    [SIM] swB     COM10  setting oscillator period to 99998 (~1000.0 Hz)


INFO    [SIM] swB     COM10  setting oscillator period to 998 (~100000.0 Hz)


INFO    [SIM] swB     COM10  setting oscillator period to 99998 (~1000.0 Hz)


skew point 1: 1 kHz (probes on out0/out1)
skew point 2: 100 kHz


[]

## M5 — current draw vs frequency (envelope mapping)

Full voltage from M3, then `ramp_frequency` with the watchdog: the point
of this matrix row is to approach (never cross) 300 mA / 100 W and map
where the envelope closes. The watchdog's step-down IS the data point.

In [12]:
ENVELOPE_TARGET_KHZ = 1000.0
try:
    f_final = ramp_frequency(swB, ENVELOPE_TARGET_KHZ, dwell_s=DWELL_S,
                             watchdog=dog)
    print(f"envelope open up to {f_final:g} kHz at this amplitude")
except WatchdogBreach as e:
    print(f"ENVELOPE EDGE: {e}")
finally:
    status, i_ma = psu2.get_psu0_output_current()
    print(f"psu2 ch0 current now: {i_ma:.1f} mA")
    swB.set_frequency_khz(RAMP_FREQ_KHZ)

INFO    [SIM] swB     COM10  setting oscillator period to 49998 (~2000.0 Hz)


INFO    [SIM] swB     COM10  setting oscillator period to 24998 (~4000.0 Hz)


INFO    [SIM] swB     COM10  setting oscillator period to 12498 (~8000.0 Hz)


INFO    [SIM] swB     COM10  setting oscillator period to 6248 (~16000.0 Hz)


INFO    [SIM] swB     COM10  setting oscillator period to 3123 (~32000.0 Hz)


...

INFO    [SIM] swB     COM10  setting oscillator period to 389 (~255754.5 Hz)


INFO    [SIM] swB     COM10  setting oscillator period to 193 (~512820.5 Hz)


INFO    [SIM] swB     COM10  setting oscillator period to 98 (~1000000.0 Hz)


INFO    [SIM] swB     COM10  setting oscillator period to 99998 (~1000.0 Hz)


envelope open up to 1000 kHz at this amplitude
psu2 ch0 current now: 70.0 mA


In [13]:
# swB block teardown: outputs down, switch to standby.
for p in (psu1, psu2):
    p.set_psu_output_voltage(p.PSU_POS, 0.0)
    p.set_psu_output_voltage(p.PSU_NEG, 0.0)
    p.set_psu_enable(False, False)
    p.set_device_enable(False)
swB.set_device_enable(False)
print("swB block parked")

INFO    [SIM] psu1    COM15  setting PSU0 output voltage to 0.000 V


INFO    [SIM] psu1    COM15  setting PSU1 output voltage to 0.000 V


INFO    [SIM] psu1    COM15  setting PSU enable to psu0=False, psu1=False


INFO    [SIM] psu1    COM15  setting device enable to False


INFO    [SIM] psu2    COM16  setting PSU0 output voltage to 0.000 V


INFO    [SIM] psu2    COM16  setting PSU1 output voltage to 0.000 V


INFO    [SIM] psu2    COM16  setting PSU enable to psu0=False, psu1=False


INFO    [SIM] psu2    COM16  setting device enable to False


INFO    [SIM] swB     COM10  setting device enable to False


swB block parked


# swA session (SWHR — high resolution)

**First task ([[cgc-sw]]): clone swB's SwitchSym RF ladder into swA's
NVM** — `config_swA.cfg` has only test/static configs. Then the fine-delay
sweep: coarse 5 ns steps, fine 11 ps steps
(`SWITCH_DELAY_FINE_SCALE`).

In [14]:
# swA bring-up (Q2 = psu3 at 300 mA; Q3/4 = psu4 at 150 mA per 023).
psu3.set_psu0_output_current(300); psu3.set_psu1_output_current(300)
psu4.set_psu0_output_current(150); psu4.set_psu1_output_current(150)
for p in (psu3, psu4):
    p.set_device_enable(True)
    p.set_psu_enable(True, True)
swA.set_device_enable(True)
dog.reset()
print("swA block armed:", dog.check() == [])

INFO    [SIM] psu3    COM17  setting PSU0 output current to 300.000 mA


INFO    [SIM] psu3    COM17  setting PSU1 output current to 300.000 mA


INFO    [SIM] psu4    COM18  setting PSU0 output current to 150.000 mA


INFO    [SIM] psu4    COM18  setting PSU1 output current to 150.000 mA


INFO    [SIM] psu3    COM17  setting device enable to True


INFO    [SIM] psu3    COM17  setting PSU enable to psu0=True, psu1=True


INFO    [SIM] psu4    COM18  setting device enable to True


INFO    [SIM] psu4    COM18  setting PSU enable to psu0=True, psu1=True


INFO    [SIM] swA     COM9   setting device enable to True


swA block armed: True


In [15]:
# RF-ladder clone scaffold: swB slot ladder -> swA NVM slots.
# Slot layout mirrors swB: 40=1 kHz ... using set_frequency_khz on
# oscillator 0, then save_current_config. AT THE BENCH: verify each
# saved slot by loading it back and checking the oscillator register.
LADDER = {40: 1, 50: 10, 60: 50, 70: 100, 80: 500, 90: 1000}  # slot: kHz
for slot, f_khz in LADDER.items():
    swA.set_frequency_khz(0, f_khz)
    swA.save_current_config(slot)
    status, reg = swA.get_oscillator_period(0)
    print(f"slot {slot:3d} <- {f_khz:5d} kHz (osc0 reg {reg})")
print("ladder written - verify with load_current_config at the bench")

INFO    [SIM] swA     COM9   setting oscillator 0 period to 99998 (~1000.0 Hz)


INFO    [SIM] swA     COM9   saving config slot 40


INFO    [SIM] swA     COM9   setting oscillator 0 period to 9998 (~10000.0 Hz)


INFO    [SIM] swA     COM9   saving config slot 50


INFO    [SIM] swA     COM9   setting oscillator 0 period to 1998 (~50000.0 Hz)


...

INFO    [SIM] swA     COM9   setting oscillator 0 period to 198 (~500000.0 Hz)


INFO    [SIM] swA     COM9   saving config slot 80


INFO    [SIM] swA     COM9   setting oscillator 0 period to 98 (~1000000.0 Hz)


INFO    [SIM] swA     COM9   saving config slot 90


slot  40 <-     1 kHz (osc0 reg 99998)
slot  50 <-    10 kHz (osc0 reg 9998)
slot  60 <-    50 kHz (osc0 reg 1998)
slot  70 <-   100 kHz (osc0 reg 998)
slot  80 <-   500 kHz (osc0 reg 198)
slot  90 <-  1000 kHz (osc0 reg 98)
ladder written - verify with load_current_config at the bench


## M3-HR — fine-delay sweep -> edge placement

Rise fine delay 0 -> 0x1FF in steps; scope: edge shift must track
11 ps/LSB. Same for fall delay on the second probe.

In [16]:
FINE_STEPS = [0x000, 0x080, 0x100, 0x180, 0x1FF]
swA.set_frequency_khz(0, RAMP_FREQ_KHZ)
for fine in FINE_STEPS:
    swA.set_switch_rise_delay_fine(0, fine)
    expect_ps = fine * swA.SWITCH_DELAY_FINE_SCALE * 1e12
    print(f"fine 0x{fine:03X} -> expected edge shift {expect_ps:7.0f} ps")
    if not SIM:
        input("  scope: measure edge shift, ENTER for next")
    dog.check()
swA.set_switch_rise_delay_fine(0, 0)

INFO    [SIM] swA     COM9   setting oscillator 0 period to 99998 (~1000.0 Hz)


INFO    [SIM] swA     COM9   setting switch 0 fine rise delay to 0


INFO    [SIM] swA     COM9   setting switch 0 fine rise delay to 128


INFO    [SIM] swA     COM9   setting switch 0 fine rise delay to 256


INFO    [SIM] swA     COM9   setting switch 0 fine rise delay to 384


INFO    [SIM] swA     COM9   setting switch 0 fine rise delay to 511


INFO    [SIM] swA     COM9   setting switch 0 fine rise delay to 0


fine 0x000 -> expected edge shift       0 ps
fine 0x080 -> expected edge shift    1408 ps
fine 0x100 -> expected edge shift    2816 ps
fine 0x180 -> expected edge shift    4224 ps
fine 0x1FF -> expected edge shift    5621 ps


0

In [17]:
# swA voltage + envelope rows reuse the same helpers via FEEDS:
DWELL_S = 0.05 if SIM else 2.0
try:
    ramp_voltage_at_1khz(swA, psu3, psu3.PSU_POS, 100.0, step_v=10.0,
                         dwell_s=DWELL_S, watchdog=dog, oscillator=0)
    f_final = ramp_frequency(swA, 500.0, dwell_s=DWELL_S, watchdog=dog,
                             oscillator=0)
    print(f"swA envelope open up to {f_final:g} kHz at 100 V")
except WatchdogBreach as e:
    print(f"swA ENVELOPE EDGE: {e}")

INFO    [SIM] swA     COM9   setting oscillator 0 period to 99998 (~1000.0 Hz)


INFO    [SIM] psu3    COM17  setting PSU0 output voltage to 10.000 V


INFO    [SIM] psu3    COM17  setting PSU0 output voltage to 20.000 V


INFO    [SIM] psu3    COM17  setting PSU0 output voltage to 30.000 V


INFO    [SIM] psu3    COM17  setting PSU0 output voltage to 40.000 V


...

INFO    [SIM] swA     COM9   setting oscillator 0 period to 1560 (~64020.5 Hz)


INFO    [SIM] swA     COM9   setting oscillator 0 period to 779 (~128041.0 Hz)


INFO    [SIM] swA     COM9   setting oscillator 0 period to 389 (~255754.5 Hz)


INFO    [SIM] swA     COM9   setting oscillator 0 period to 198 (~500000.0 Hz)


swA envelope open up to 500 kHz at 100 V


# Emergency shutdown / end of session

PSUs down FIRST, then switches — always.

In [18]:
def emergency_shutdown():
    for name, p in [("psu1", psu1), ("psu2", psu2), ("psu3", psu3), ("psu4", psu4)]:
        try:
            p.set_psu_output_voltage(p.PSU_POS, 0.0)
            p.set_psu_output_voltage(p.PSU_NEG, 0.0)
            p.set_psu_enable(False, False)
            p.set_device_enable(False)
            print(f"{name}: outputs 0 V, disabled")
        except Exception as e:
            print(f"{name}: SHUTDOWN FAILED: {e}")
    for name, s in [("swA", swA), ("swB", swB)]:
        try:
            s.set_device_enable(False)
            print(f"{name}: disabled")
        except Exception as e:
            print(f"{name}: SHUTDOWN FAILED: {e}")

emergency_shutdown()

INFO    [SIM] psu1    COM15  setting PSU0 output voltage to 0.000 V


INFO    [SIM] psu1    COM15  setting PSU1 output voltage to 0.000 V


INFO    [SIM] psu1    COM15  setting PSU enable to psu0=False, psu1=False


INFO    [SIM] psu1    COM15  setting device enable to False


INFO    [SIM] psu2    COM16  setting PSU0 output voltage to 0.000 V


...

INFO    [SIM] psu4    COM18  setting PSU enable to psu0=False, psu1=False


INFO    [SIM] psu4    COM18  setting device enable to False


INFO    [SIM] swA     COM9   setting device enable to False


INFO    [SIM] swB     COM10  setting device enable to False


psu1: outputs 0 V, disabled
psu2: outputs 0 V, disabled
psu3: outputs 0 V, disabled
psu4: outputs 0 V, disabled
swA: disabled
swB: disabled


In [19]:
for name, dev in ALL:
    dev.disconnect()
print("all disconnected - session over. Debrief into the guide TODAY.")

INFO    [SIM] psu1    COM15  disconnected


INFO    [SIM] psu2    COM16  disconnected


INFO    [SIM] psu3    COM17  disconnected


INFO    [SIM] psu4    COM18  disconnected


INFO    [SIM] swA     COM9   disconnected


INFO    [SIM] swB     COM10  disconnected


all disconnected - session over. Debrief into the guide TODAY.
